# 02 - Build the embedding dataset 🧬

This notebook downloads both ONNX encoders, extracts embeddings, creates fixed train/validation/test splits, and uploads a Hugging Face dataset.

Rows are stored in CSV files. High-dimensional feature matrices are stored in compressed NPZ files beside the CSV files.

In [1]:
WK_DIR = "/content/"

In [2]:
%cd {WK_DIR}
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
!git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
%pip install -e ".[runtime]" "huggingface-hub>=0.30"

/content
[Errno 2] No such file or directory: '/content/Protein-Ligand-Affinity-Prediction-Using-LLM'
/content
Cloning into 'Protein-Ligand-Affinity-Prediction-Using-LLM'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 100 (delta 30), reused 97 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 218.37 KiB | 1.57 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/Protein-Ligand-Affinity-Prediction-Using-LLM
Obtaining file:///content/Protein-Ligand-Affinity-Prediction-Using-LLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [15]:
import os, subprocess
import kagglehub
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
from pathlib import Path
os.sys.path.insert(0,f'{WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM/src')

HF_USER = userdata.get('HF_USER')
token = userdata.get('HF_TOKEN')
login(token=token)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
PRO_REPO = f'{HF_USER}/prollama-affinity-onnx'
MOL_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
DATASET_REPO = f'{HF_USER}/protein-compound-affinity-embeddings'
ROOT = Path(f'{WK_DIR}/protein_affinity')
RAW_DATA = ROOT / 'data/train.csv'
CACHE = ROOT / 'embedding_cache'
DATASET_OUT = ROOT / 'embedding_dataset'
CACHE.mkdir(parents=True, exist_ok=True)
RAW_DATA.parent.mkdir(parents=True, exist_ok=True)
DATASET_OUT.mkdir(parents=True, exist_ok=True)
kagglehub.competition_download('protein-compound-affinity',output_dir = ROOT/'data',force_download=True)
assert RAW_DATA.exists(), f'Missing: {RAW_DATA}'
PRO_ONNX = Path(snapshot_download(PRO_REPO, token=token))
MOL_ONNX = Path(snapshot_download(MOL_REPO, token=token))

100%|██████████| 22.4M/22.4M [00:00<00:00, 205MB/s]

Extracting files...


## Inspect the source data

In [16]:
from affinity.data import load_dataset, profile_dataset
frame = load_dataset(RAW_DATA)
profile_dataset(frame)

DatasetProfile(rows=263583, unique_proteins=2665, unique_compounds=196029, unique_pairs=256296, protein_length_min=66, protein_length_max=1484, protein_length_mean=582.7472371131674, smiles_length_min=2, smiles_length_max=100, smiles_length_mean=56.88849812013673, label_min=2.0, label_max=11.0, label_mean=6.33967399597168, label_std=1.4751832485198975, duplicate_pairs=7287)

## Extract ONNX embeddings

Protein embeddings are stored in one file. Molecule embeddings are sharded so extraction can resume after a disconnect.

In [ ]:
subprocess.run([
    'affinity-extract-onnx',
    '--data', str(RAW_DATA),
    '--prollama-onnx', str(PRO_ONNX),
    '--mol-llama-onnx', str(MOL_ONNX),
    '--protein-output', str(CACHE / 'prollama_embeddings.npz'),
    '--molecule-output', str(CACHE / 'mol_llama'),
    '--protein-max-length', '1536',
    '--protein-batch-size', '1',
    '--molecule-shard-size', '1000',
], check=True)

## Create fixed train, validation and test files

In [ ]:
subprocess.run([
    'affinity-prepare-dataset',
    '--data', str(RAW_DATA),
    '--protein-embeddings', str(CACHE / 'prollama_embeddings.npz'),
    '--molecule-embeddings', str(CACHE / 'mol_llama'),
    '--output', str(DATASET_OUT),
    '--split-strategy', 'cold_protein',
    '--seed', '42',
], check=True)
print(sorted(path.name for path in DATASET_OUT.iterdir()))

In [ ]:
import pandas as pd, numpy as np, json
for split in ['train', 'validation', 'test']:
    rows = pd.read_csv(DATASET_OUT / f'{split}.csv')
    features = np.load(DATASET_OUT / f'{split}_features.npz')['features']
    print(split, 'rows=', len(rows), 'features=', features.shape)
print(json.loads((DATASET_OUT / 'dataset_metadata.json').read_text()))

## Upload the embedding dataset

In [ ]:
UPLOAD = False
if UPLOAD:
    api = HfApi(token=token)
    api.create_repo(DATASET_REPO, repo_type='dataset', exist_ok=True)
    api.upload_large_folder(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        folder_path=str(DATASET_OUT),
    )
    print('Uploaded:', DATASET_REPO)